# 01 — Preprocessing & Song Manifest

Build a song-level manifest of available mel files, join official **split-0** membership, and write `dataset/song_manifest.csv`.

**Requires:** outputs of `00_kaggle_data_download` (or attached cache dataset).


## Attach kernel output from notebook 00 (`dnn-download-data-1`)

Do **not** run this inside the Kaggle preprocessing notebook:

```bash
kaggle kernels output thevifernando/dnn-download-data-1 -p /path/to/dest
```

That command is only for your **laptop** (Kaggle CLI). On Kaggle you **attach** the kernel output as input data.

### On Kaggle (do this)

1. Open this preprocessing notebook.
2. Right sidebar → **Add Input** (sometimes labelled **Add Data**).
3. Choose **Your notebooks** / **Notebook Output**.
4. Select **`dnn-download-data-1`** (`thevifernando/dnn-download-data-1`).
5. Kaggle mounts it at:

```text
/kaggle/input/dnn-download-data-1/
```

(usually with `MTG_Instrument/` inside it).

6. Run the next cell — it should print `Kernel output attached: True`.

### On your PC only (optional)

```bash
kaggle kernels output thevifernando/dnn-download-data-1 -p ./from_00
```

That downloads files locally. It does **not** feed the Kaggle preprocessing notebook.

In [ ]:
from pathlib import Path

# Slug of notebook 00 after you click Add Input → Notebook Output
KERNEL_SLUG = "dnn-download-data-1"
KERNEL_OUTPUT = Path("/kaggle/input") / KERNEL_SLUG

print("Looking for:", KERNEL_OUTPUT)
print("Kernel output attached:", KERNEL_OUTPUT.exists())

inp = Path("/kaggle/input")
if inp.exists():
    print("\nFolders currently under /kaggle/input:")
    for p in sorted(inp.iterdir()):
        print(" ", p)
else:
    print("/kaggle/input does not exist (not on Kaggle?)")

if KERNEL_OUTPUT.exists():
    print("\nTop-level contents of the attached kernel output:")
    for p in sorted(KERNEL_OUTPUT.rglob("*")):
        if p.is_file() and p.suffix in {".tsv", ".csv", ".json", ".npy"}:
            print(" ", p.relative_to(KERNEL_OUTPUT))
            if p.suffix == ".npy":
                break  # don't print thousands of npy paths
        elif p.is_dir() and len(p.relative_to(KERNEL_OUTPUT).parts) <= 3:
            print(" [dir]", p.relative_to(KERNEL_OUTPUT))
else:
    print(
        "\n❌ Not attached yet.\n"
        "   Add Input → Notebook Output → dnn-download-data-1\n"
        "   Then re-run this cell."
    )

In [ ]:
from pathlib import Path
import os, json, random, shutil
import numpy as np
import pandas as pd

# --- Point preprocessing at kernel 00 output ---
KERNEL_SLUG = "dnn-download-data-1"
KERNEL_OUTPUT = Path("/kaggle/input") / KERNEL_SLUG
WORKING_ROOT = Path("/kaggle/working/MTG_Instrument")

# Typical layouts after Add Input:
#   /kaggle/input/dnn-download-data-1/MTG_Instrument/...
#   /kaggle/input/dnn-download-data-1/annotations/...
candidates = [
    KERNEL_OUTPUT / "MTG_Instrument",
    KERNEL_OUTPUT,
    Path("/kaggle/working/MTG_Instrument"),
]
INPUT_ROOT = next(
    (p for p in candidates if (p / "annotations").exists() or (p / "dataset").exists()),
    None,
)

USE_CACHED_INPUT = INPUT_ROOT is not None
if USE_CACHED_INPUT:
    ROOT = INPUT_ROOT
    print("Using attached kernel output at", ROOT)
else:
    ROOT = WORKING_ROOT
    print("No kernel output found — using", ROOT)
    print("Add Input → dnn-download-data-1, or run notebook 00 in this same session.")

MEL_DIR = ROOT / "dataset" / "logmel_songs"
ANN_DIR = ROOT / "annotations"
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = WORKING_ROOT / "dataset" / "song_manifest.csv"  # always write to working (input is read-only)

for p in [WORKING_ROOT / "dataset", FEAT_DIR, CKPT_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("ROOT    =", ROOT)
print("MEL_DIR =", MEL_DIR, "exists=", MEL_DIR.exists())
print("ANN_DIR =", ANN_DIR, "exists=", ANN_DIR.exists())
print("train TSV exists=", (ANN_DIR / "splits" / "split-0" / "autotagging_genre-train.tsv").exists())
print("MANIFEST will be written to", MANIFEST)


In [ ]:
from pathlib import Path
import re

def normalize_track_id(raw: str) -> str | None:
    """MTG track ids are 7-digit zero-padded (e.g. track_0000948 → 0000948)."""
    m = re.search(r"(\d+)", str(raw))
    if not m:
        return None
    return f"{int(m.group(1)):07d}"

def track_id_from_path(p: Path) -> str | None:
    return normalize_track_id(p.stem)

rows = []
for p in MEL_DIR.rglob("*.npy"):
    tid = track_id_from_path(p)
    if tid is None:
        continue
    rows.append({
        "song_id": tid,
        "mel_path": str(p.relative_to(ROOT)) if str(p).startswith(str(ROOT)) else str(p),
        "mel_abs": str(p),
        "nbytes": p.stat().st_size,
    })

mel_df = pd.DataFrame(rows).drop_duplicates("song_id")
print("Unique songs with mel:", len(mel_df))
mel_df.head()


In [ ]:
def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    """Load official split-0 track IDs from MTG autotagging_{subset}-{split}.tsv."""
    name = f"autotagging_{subset}-{split}.tsv"
    candidates = [
        ANN_DIR / "splits" / "split-0" / name,
        ANN_DIR / "splits" / "split-0" / f"{split}.tsv",
        ANN_DIR / name,
        ANN_DIR / f"{split}.tsv",
    ]
    # also search the attached kernel output (read-only /kaggle/input)
    kernel = Path("/kaggle/input/dnn-download-data-1")
    if kernel.exists():
        candidates.extend(kernel.rglob(name))
        candidates.extend(kernel.rglob(f"{split}.tsv"))

    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(
            f"No split file for {subset}/{split}.\n"
            "Attach kernel output: Add Input → Notebook Output → dnn-download-data-1\n"
            f"Tried: {candidates[:8]}"
        )
    df = pd.read_csv(path, sep="\t")
    col = "TRACK_ID" if "TRACK_ID" in df.columns else df.columns[0]
    ids = set()
    for v in df[col].astype(str):
        tid = normalize_track_id(v)
        if tid:
            ids.add(tid)
    print(split, "ids:", len(ids), "from", path)
    return ids

train_ids = load_split_ids("train")
val_ids = load_split_ids("validation")
test_ids = load_split_ids("test")

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids), "VALIDATION must not intersect TEST"
print("Split leakage check: OK")


In [ ]:
def split_of(sid: str) -> str:
    if sid in train_ids:
        return "train"
    if sid in val_ids:
        return "validation"
    if sid in test_ids:
        return "test"
    return "unused"

mel_df["split"] = mel_df["song_id"].map(split_of)
print(mel_df["split"].value_counts())

# Keep only official split-0 songs present in our shard subset
manifest = mel_df[mel_df["split"] != "unused"].copy()
manifest.to_csv(MANIFEST, index=False)
print("Wrote", MANIFEST, "rows=", len(manifest))
manifest.head()


In [ ]:
# Optional: peek one mel shape
sample = np.load(manifest.iloc[0]["mel_abs"])
print("example shape:", sample.shape, "dtype:", sample.dtype)
(RESULTS_DIR / "01_manifest_summary.json").write_text(json.dumps({
    "n_manifest": int(len(manifest)),
    "split_counts": manifest["split"].value_counts().to_dict(),
    "example_shape": list(sample.shape),
}, indent=2))
